# CS 195: Natural Language Processing
## Fine-Tuning a Transformer Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F6_4_FineTuningTransformers.ipynb)


In [ ]:
# I did have to run this on Colab
import sys
!{sys.executable} -m pip install -U transformers datasets accelerate trl peft torchao

## References

- [SLP: Post-training: Instruction Tuning, Alignment, and Test-Time Compute, Chapter 9 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/8.pdf)
- [Hugging Face PEFT LoRA conceptual guide](https://huggingface.co/docs/peft/main/en/conceptual_guides/lora)
- [LoRA paper: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

- [Hugging Face TRL `SFTTrainer`](https://huggingface.co/docs/trl/main/en/sft_trainer)
- [Hugging Face PEFT LoRA guide](https://huggingface.co/docs/peft/en/developer_guides/lora)



## Fine-Tuning

A pretrained transformer model has already learned a lot from large amounts of general text. Fine-tuning means we continue training that existing model on a smaller, more specific dataset.

<div>
    <center>
    <img src="images/fine_tuning.png">
    </center>
</div>

image source: SLP Fig. 9.1, https://web.stanford.edu/~jurafsky/slp3/9.pdf


The goal is not usually to teach the model language from scratch. The goal is to adapt what it already knows so it performs better for a particular purpose.

Fine-tuning is useful when we want to change or improve things like:
- the kind of task the model performs
- the format of its answers
- the style or tone of its responses
- how reliably it follows a particular instruction pattern
- performance on a specific domain or dataset

Examples:
- fine-tune a sentiment model on movie reviews
- fine-tune a medical classifier on clinical notes
- fine-tune a small chat model to answer in a course-assistant style
- fine-tune a model to produce outputs in a strict JSON format
- fine-tune a model to return names and parameters of functions that an agent should call



## What Kind of Fine-Tuning Are We Doing?

In this notebook se are doing **supervised fine-tuning** (often shortened to **SFT**).

We are going to generate a training data set using our Drake course schedule data from back when we did RAG. 
* note that this won't likely give us better answers than if we had just used RAG
* but you will see the idea - the model will now inherently have some of the data from our dataset and will be ready to answer in the style we've provided
* in real life you'd probably want to combine this style of fine-tuning with a RAG system

That means we give the model examples of the behavior we want:

```text
User: Who teaches CS 130?
Assistant: CS 130 is taught by Eric Manley.
```

Under the hood, the model is still doing the same basic language-modeling task:
- read the previous tokens
- predict the next token
- update weights to make the correct answer tokens more likely

So this is not a new kind of neural network. It is a new use of the same next-token prediction training objective.


## Full Fine-Tuning vs. PEFT

A pretrained transformer has many learned weight matrices. In **full fine-tuning**, we update all or most of those weights.

That can work, but it is expensive:
- more GPU memory
- more training time
- larger saved checkpoints
- easier to overfit on a small dataset

**PEFT** means **Parameter-Efficient Fine-Tuning**. The idea is to train only a small number of new parameters while keeping most of the original model frozen.

For today's lab, we'll use a form of PEFT so it will finish in a reasonable time


## LoRA

**LoRA** stands for **Low-Rank Adaptation**.

A transformer is not one big matrix. It is a stack of transformer blocks, and each block contains several learned pieces: attention projections, feed-forward layers, normalization parameters, and more.

LoRA does **not** add one new layer at the end of the model.
It also does **not** simply treat the transformer as a frozen feature extractor.

Instead, LoRA usually goes **inside existing transformer layers** and adds small trainable adapter updates to selected weight matrices.

So LoRA is changing how the model uses attention, but it does this by training a small number of new adapter parameters while leaving the original pretrained weights mostly frozen.


## LoRA as a Small Side Path Inside Attention

Think back to the query-key-value picture from F6_3. Inside a transformer block, the model uses learned projection layers to create query, key, and value vectors from each token representation.

<div>
    <center>
    <img src="images/query_key_value.png" width=650>
    </center>
</div>

image source: SLP Fig. 8.5, https://web.stanford.edu/~jurafsky/slp3/8.pdf


With LoRA, the original projection stays frozen, and we add a small trainable side path. The Hugging Face diagram below shows this idea for one learned weight matrix.

<div>
    <center>
    <img src="images/lora_diagram.png" width=650>
    </center>
</div>

image source: https://huggingface.co/docs/peft/main/en/conceptual_guides/lora

The diagram shows one frozen pretrained matrix `W` and a small trainable LoRA update.

During normal use, a layer might compute something like:

```text
h = W x
```

where:
- `x` is the input vector to that projection layer
- `W` is the original pretrained weight matrix
- `h` is the output vector

With LoRA, we keep `W` frozen and add a learned update:

```text
h = W x + B A x
```

The pieces mean:

- `W`: the original pretrained weight matrix. It stays frozen during LoRA fine-tuning.
- `A` and `B`: the small LoRA adapter matrices with sizes like `[d,r]` and `[r,d]`. These are the trainable part. 
- `r`: the small middle dimension, called the **rank**. When `r` is small, like `8` or `16`, that is much smaller than updating the whole original matrix.
- $A = \mathcal{N}(0, \sigma^2)$: `A` starts with small random values sampled from a normal distribution.
- `B = 0`: `B` starts as all zeros.

That last point is important. Since `B` starts at zero, the LoRA update `B A x` starts out as zero too.

So at the beginning of training:

```text
h = W x + 0
```



which means the model initially behaves just like the original pretrained model. Then training gradually changes `A` and `B`, allowing the adapter path to learn a useful correction.

You can apply this to any of the layers like the matrix that produces the query, key, and value vectors or the vector that projects the attention output (step 7 in the diagram above - or the one that combines all of the outputs from multi-headed version) to the hidden representation.



## Imports and Device

This lab is designed for Colab with a GPU. It may run on CPU, but training will be much slower.


In [1]:
import json
import random
from pathlib import Path
import requests

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)


Using device: mps


## Choose a Small Instruction Model

`Qwen/Qwen2.5-0.5B-Instruct` is a good default if you have a Colab GPU.

If you need something smaller, try:

```python
checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"
```


In [2]:
checkpoint = "Qwen/Qwen2.5-0.5B-Instruct"
#checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.to(device)

print('Loaded:', checkpoint)
print('Vocab size:', len(tokenizer))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct
Vocab size: 151665


## Helper Function: Chat with the Model

This uses the tokenizer's chat template so the prompt is formatted the way the model expects.


In [3]:
def ask_model(question, model=model, tokenizer=tokenizer, max_new_tokens=120):
    messages = [
        {
            'role': 'system',
            'content': 'You are a helpful academic advising assistant. Answer clearly and concisely.'
        },
        {
            'role': 'user',
            'content': question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


## Before Fine-Tuning: Try a Few Course Questions

The model has probably not memorized Drake's Fall 2025 course schedule. That is useful: we can see what it does before training.


In [4]:
before_questions = [
    'Who teaches CS 130?',
    'Where does CS 130 meet?',
    'What are the prerequisites for CS 130?',
]

for q in before_questions:
    print('=' * 80)
    print('QUESTION:', q)
    print('ANSWER:')
    print(ask_model(q))
    print()


QUESTION: Who teaches CS 130?
ANSWER:


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CS 130 is taught by Dr. Jane Smith, an Associate Professor in the Computer Science Department at XYZ University.

QUESTION: Where does CS 130 meet?
ANSWER:
CS 130 typically meets in the Math Building, Room 245, at the University of California, Berkeley campus.

QUESTION: What are the prerequisites for CS 130?
ANSWER:
The prerequisite for CS 130, which is typically CS 125 or equivalent, is either CS 125 (Introduction to Computer Science) or CS 140 (Data Structures). These courses provide foundational knowledge in computer science concepts that will be useful in CS 130.



## Discuss

Why is the model responding this way?

## Load the Course Information Data

The raw data already exists in this repository. Each record describes one course section.


In [5]:

# uncomment these if you want to load locally
# data_path = Path('data/f25_course_information.json')
# with open(data_path) as f:
#     courses = json.load(f)

url = "https://raw.githubusercontent.com/ericmanley/S26-CS195NLP/refs/heads/main/data/f25_course_information.json"

response = requests.get(url)
response.raise_for_status()

courses = response.json()

print('number of course records:', len(courses))
print(courses[0].keys())
print(courses[0])


number of course records: 1175
dict_keys(['id', 'term', 'course_number', 'subject', 'title', 'course_search_url', 'prereq', 'description', 'credit_hours', 'faculty', 'attributes', 'location', 'times', 'filename'])
{'id': 213644, 'term': 'Fall 2025', 'course_number': 'ACCT 041', 'subject': 'Accounting', 'title': 'INTRODUCTION TO FINANCIAL ACCOUNTING', 'course_search_url': 'https://catalog.drake.edu/course-search/?details&srcdb=2024&code=ACCT%20041', 'prereq': 'Prerequisite(s): None', 'description': '\n\n    The elements of the financial statements, accounting for deferrals, the double-entry accounting system, internal control and cash, receivables and payables, inventory, operational assets, long-term debt, equity transactions, income measurement, and comprehensive treatment of the balance sheet, the income statement and the statement of cash flows.  Financial statement analysis will be integrated throughout the course.\n    \n\n', 'credit_hours': None, 'faculty': ['Joyce Njoroge'], 'at

## Build a Small Instruction Dataset Automatically

To keep this lab moving, we will generate simple question-answer pairs from structured fields in the JSON file.

This is not a perfect dataset, but it lets us practice the fine-tuning workflow without hand-writing hundreds of examples.


In [6]:
def clean_text(value):
    if value is None:
        return 'Not listed'
    if isinstance(value, list):
        return ', '.join(str(x).strip() for x in value if str(x).strip()) or 'Not listed'
    return ' '.join(str(value).split()) or 'Not listed'


def course_label(course):
    return f"{course.get('course_number', 'Unknown course')} ({course.get('title', 'Untitled')})"


def examples_for_course(course):
    label = course_label(course)
    number = clean_text(course.get('course_number'))
    title = clean_text(course.get('title'))
    faculty = clean_text(course.get('faculty'))
    times = clean_text(course.get('times'))
    location = clean_text(course.get('location'))
    prereq = clean_text(course.get('prereq'))
    description = clean_text(course.get('description'))
    attributes = clean_text(course.get('attributes'))

    return [
        {
            'question': f'What is the title of {number}?',
            'answer': f'The title of {number} is {title}.'
        },
        {
            'question': f'Who teaches {number}?',
            'answer': f'{label} is taught by {faculty}.'
        },
        {
            'question': f'When does {number} meet?',
            'answer': f'{label} meets at {times}.'
        },
        {
            'question': f'Where does {number} meet?',
            'answer': f'{label} meets in {location}.'
        },
        {
            'question': f'What are the prerequisites for {number}?',
            'answer': f'The prerequisites for {label} are: {prereq}.'
        },
        {
            'question': f'What attributes does {number} have?',
            'answer': f'{label} has these attributes: {attributes}.'
        },
        {
            'question': f'Describe {number}.',
            'answer': f'{label}: {description}'
        },
    ]


# Use a subset so training stays quick. Increase this later if you want.
course_subset = courses[:120]
qa_examples = []
for course in course_subset:
    qa_examples.extend(examples_for_course(course))

random.shuffle(qa_examples)
print('number of generated examples:', len(qa_examples))
qa_examples[:3]


number of generated examples: 840


[{'question': 'What attributes does ART 175 have?',
  'answer': 'ART 175 (SENIOR STUDIO ART CAPSTONE I) has these attributes: Not listed.'},
 {'question': 'What are the prerequisites for ART 063?',
  'answer': 'The prerequisites for ART 063 (PAINTING I) are: Prerequisite(s): None.'},
 {'question': 'Who teaches ATHL 283?',
  'answer': 'ATHL 283 (ATHLETIC TRAINING SEMINAR III: POST PROFESSIONAL PREPARATION) is taught by Nate Newman.'}]

## Format Examples as Chat Conversations

Instruction-tuned models are trained on conversations. We'll format each training example as:

```text
system: You are a helpful academic advising assistant.
user: question
assistant: answer
```


In [7]:
def format_as_chat_text(example):
    messages = [
        {
            'role': 'system',
            'content': 'You are a helpful academic advising assistant. Answer using the course information you learned during training.'
        },
        {
            'role': 'user',
            'content': example['question']
        },
        {
            'role': 'assistant',
            'content': example['answer']
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted_examples = [{'text': format_as_chat_text(ex), **ex} for ex in qa_examples]

split = int(0.85 * len(formatted_examples))
train_rows = formatted_examples[:split]
eval_rows = formatted_examples[split:]

train_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)

print(train_dataset[0]['text'])
print('train examples:', len(train_dataset))
print('eval examples:', len(eval_dataset))


<|im_start|>system
You are a helpful academic advising assistant. Answer using the course information you learned during training.<|im_end|>
<|im_start|>user
What attributes does ART 175 have?<|im_end|>
<|im_start|>assistant
ART 175 (SENIOR STUDIO ART CAPSTONE I) has these attributes: Not listed.<|im_end|>

train examples: 714
eval examples: 126


## Pick a Few Held-Out Questions

We'll use some eval-set questions before and after fine-tuning.


In [8]:
heldout_questions = [row['question'] for row in eval_rows[:5]]
heldout_answers = [row['answer'] for row in eval_rows[:5]]

for q, a in zip(heldout_questions, heldout_answers):
    print('QUESTION:', q)
    print('EXPECTED:', a)
    print()


QUESTION: Describe BIO 012L.
EXPECTED: BIO 012L (GENERAL/PRE-PROFESSIONAL BIOLOGY I LAB): Co-requisite lab for BIO 012.

QUESTION: What attributes does ANTH 002 have?
EXPECTED: ANTH 002 (INTRO CULTURAL ANTHROPOLOGY) has these attributes: Global and Cultural Understand.

QUESTION: Describe ACTS 131.
EXPECTED: ACTS 131 (INTRODUCTION TO PROBABILITY I): An introduction to probability concepts; including definition of probability; independence; conditional probability; random variables; specific discrete and continuous probability distributions; multivariate random variables; moments and moment generating functions; functions of random variables; sampling distributions and central limit theorem.

QUESTION: What is the title of ART 145?
EXPECTED: The title of ART 145 is EXPERIMENTAL DRAWING.

QUESTION: What is the title of BCMB 199?
EXPECTED: The title of BCMB 199 is BCMB RESEARCH.



## What Is TRL and `SFTTrainer`?

**TRL** is a Hugging Face library for training language models with common post-training workflows.

`SFTTrainer` is a trainer designed for supervised fine-tuning. It handles several pieces we would otherwise have to write ourselves:

- tokenizing formatted training examples
- batching examples together
- running the training loop
- computing the language-modeling loss
- saving/checkpointing training outputs
- optionally attaching PEFT/LoRA adapters

In other words, `SFTTrainer` is doing the same kind of work as our PyTorch training loops, but packaged for transformer fine-tuning.

This is a very practical, reusable skill to learn


## Reading the LoRA Configuration

In the next cell:

- `r=8`: the small bottleneck/rank from the diagram. Smaller means fewer trainable parameters.
- `lora_alpha=16`: scales the LoRA update before it is added back into the frozen path.
- `lora_dropout=0.05`: adds a little regularization while training.
- `task_type='CAUSAL_LM'`: tells PEFT this is a decoder-style next-token model.
- `target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']`: attach LoRA adapters inside the attention projection layers.

Those projection layers connect directly to what we studied in the query-key-value diagram.


In [9]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = SFTConfig(
    output_dir='f6_4_course_assistant_lora',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='no',
    eval_strategy='no',
    max_length=512,
    report_to='none',
    fp16=torch.cuda.is_available(),
)


## Fine-Tune

This may take a few minutes on a Colab GPU.

If you run out of memory, try:
- switching to `HuggingFaceTB/SmolLM2-135M-Instruct`
- reducing `max_length`
- reducing `course_subset`
- reducing `per_device_train_batch_size`


In [10]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()


Adding EOS to train dataset:   0%|          | 0/714 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/714 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/000794593/Library/Python/3.12/lib/python/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.494392
20,2.088106
30,1.435315
40,1.251500


TrainOutput(global_step=45, training_loss=1.970392598046197, metrics={'train_runtime': 1899.9104, 'train_samples_per_second': 0.376, 'train_steps_per_second': 0.024, 'total_flos': 132360033180672.0, 'train_loss': 1.970392598046197})

## After Fine-Tuning: Ask the Held-Out Questions Again

This is a qualitative check. We are looking for whether the model has become more likely to answer in the course-assistant style and use the course-specific facts.


In [11]:
for q, expected in zip(heldout_questions, heldout_answers):
    print('=' * 80)
    print('QUESTION:', q)
    print('EXPECTED:')
    print(expected)
    print()
    print('MODEL ANSWER:')
    print(ask_model(q, model=model, tokenizer=tokenizer))
    print()


QUESTION: Describe BIO 012L.
EXPECTED:
BIO 012L (GENERAL/PRE-PROFESSIONAL BIOLOGY I LAB): Co-requisite lab for BIO 012.

MODEL ANSWER:
BIO 012L (INTRODUCTION TO BIOLOGY) is an introductory course in biology that covers fundamental concepts of biological systems, including the structure and function of cells, organisms, ecosystems, and genetics. It provides students with a foundational understanding of biological principles and processes.

QUESTION: What attributes does ANTH 002 have?
EXPECTED:
ANTH 002 (INTRO CULTURAL ANTHROPOLOGY) has these attributes: Global and Cultural Understand.

MODEL ANSWER:
ANTH 002 (INTRODUCTION TO NATURAL HISTORY) has the following attributes:
- Introduction to natural history
- Course number: 002
- Credit hours: 3
- Department: Anthropology
- Term: Fall
- Instructor: None specified
- Location: None specified
- Prerequisites: None specified
- Catalog Description: This course introduces students to the study of human origins, including the fossil record, pale

## Try Your Own Questions

Ask questions about courses that appear in the training subset.

Remember: this model is not connected to the JSON file at inference time. If it answers correctly, that information is coming from the fine-tuned weights or from prior model knowledge.


In [12]:
question = 'Who teaches CS 130?'
print(ask_model(question, model=model, tokenizer=tokenizer))


CS 130 is taught by Dr. David A. Karger.


## Discussion: What Did Fine-Tuning Actually Do?

Discuss:
1. Did the model improve on the held-out questions?
2. Did it learn facts, format, tone, or all three?
3. Where did it still hallucinate?
4. Would RAG be safer for this task? Why?
5. What kinds of tasks are better suited to fine-tuning than RAG?

A useful distinction:
- Use **fine-tuning** when you want to change behavior, format, style, or a repeated task pattern.
- Use **RAG** when you need accurate answers from a changing external knowledge source.
- Use both when you need a model that follows a specialized behavior and uses external context.


## Applied Exploration

Choose one or more of these as part of your exploration. 

1. Increase the number of course records used for training. Does performance improve?
2. Change the generated questions to make them more varied or fit with a different style.
3. Create 20 hand-written high-quality examples and compare them with the automatically generated examples.
4. Try the smaller SmolLM2 model and compare training speed and answer quality.
5. Try a different narrow task, such as formatting course information as JSON or recommending courses based on a student request.

Report what you changed, what you expected, and what happened.
